In [1]:
import numpy as np

In [2]:
data = np.load('D:\\finalproject\\_temp_batch_1.npy', allow_pickle=True)

In [3]:
data2= np.load('D:\\finalproject\\Part 1- Train data - images.npy', allow_pickle=True)

In [4]:
data3 = np.load('D:\\finalproject\\_temp_batch_2.npy', allow_pickle=True)

In [5]:
data4 = np.load('D:\\finalproject\\_temp_batch_3.npy', allow_pickle=True)

In [6]:
data5 = np.load('D:\\finalproject\\_temp_batch_5.npy', allow_pickle=True)

In [7]:
data6 = np.load('D:\\finalproject\\_temp_batch_6.npy', allow_pickle=True)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
def retrive_image(data,index):
    img = data[index][0]
    labels = data[index][1]
    height, width, _ = img.shape

    fig, ax = plt.subplots()
    ax.imshow(img)

    # 2. Loop through labels to draw the boxes/points
    for label in labels:
        p1 = label['points'][0]
        p2 = label['points'][1]
        
        # Convert normalized coordinates to pixel coordinates
        x1, y1 = p1['x'] * width, p1['y'] * height
        x2, y2 = p2['x'] * width, p2['y'] * height
        
        # Draw a rectangle (bounding box)
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)

    plt.show()


In [8]:
new_data=np.delete(data,17,axis=0)

In [9]:
data = np.concatenate((data2, new_data , data3,data4,data5,data6), axis=0)

In [10]:
data.shape

(10408, 2)

In [11]:
import tensorflow as tf

In [12]:
import numpy as np
from PIL import Image
import cv2
from scipy.ndimage import gaussian_filter
from sklearn.model_selection import train_test_split

In [ ]:
data = result.copy()

In [ ]:
data.shape

In [16]:
from sklearn.model_selection import train_test_split

TARGET_SIZE = (256, 256)

def _rotate_tensor(tensor, angle):
    """Rotate a 3D tensor (H, W, C) by angle radians."""
    cos_a = tf.cos(angle)
    sin_a = tf.sin(angle)
    # Affine transform expects [a0 a1 a2 b0 b1 b2 c0 c1] for projective
    h = tf.cast(tf.shape(tensor)[0], tf.float32)
    w = tf.cast(tf.shape(tensor)[1], tf.float32)
    cx, cy = w / 2.0, h / 2.0
    transform = [
        cos_a, -sin_a, cx - cx * cos_a + cy * sin_a,
        sin_a, cos_a, cy - cx * sin_a - cy * cos_a,
        0.0, 0.0,
    ]
    tensor = tf.expand_dims(tensor, 0)
    tensor = tf.raw_ops.ImageProjectiveTransformV3(
        images=tensor,
        transforms=tf.reshape(transform, [1, 8]),
        output_shape=tf.shape(tensor)[1:3],
        interpolation="BILINEAR",
        fill_mode="NEAREST",
        fill_value=0.0,
    )
    return tf.squeeze(tensor, 0)


def augment(image, mask):
    """Apply random augmentations to an image-mask pair (same transforms)."""
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    # Random brightness/contrast on image only
    image = tf.image.random_brightness(image, 0.15)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.clip_by_value(image, 0.0, 1.0)

    # Random rotation (small angles via affine)
    angle = tf.random.uniform((), -15.0, 15.0) * (np.pi / 180.0)
    image = _rotate_tensor(image, angle)
    mask = _rotate_tensor(mask, angle)

    # Re-binarize mask after rotation interpolation
    mask = tf.cast(mask >= 0.5, tf.float32)
    return image, mask

def annotations_to_mask(annotations, img_height, img_width):
    """Convert face bounding-boxes to elliptical masks with smoothing."""
    mask = np.zeros((img_height, img_width), dtype=np.uint8)
    for ann in annotations:
        pts = ann["points"]
        x1, y1 = int(pts[0]["x"] * img_width), int(pts[0]["y"] * img_height)
        x2, y2 = int(pts[1]["x"] * img_width), int(pts[1]["y"] * img_height)
        x1, x2 = min(x1, x2), max(x1, x2)
        y1, y2 = min(y1, y2), max(y1, y2)
        center = ((x1 + x2) // 2, (y1 + y2) // 2)
        axes = ((x2 - x1) // 2, (y2 - y1) // 2)
        cv2.ellipse(mask, center, axes, 0, 0, 360, 1, -1)
    mask = gaussian_filter(mask.astype(np.float32), sigma=1.0)
    return (mask > 0.5).astype(np.uint8)


# Split indices only — no image data in memory
train_idx, val_idx = train_test_split(range(len(data)), test_size=0.2, random_state=42)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")



def make_generator(indices):
    def gen():
        for i in indices:
            img, anns = data[i][0], data[i][1]
            if img.ndim == 3 and img.shape[2] == 4:
                img = img[:, :, :3]

            # Create mask at original resolution
            mask = annotations_to_mask(anns, img.shape[0], img.shape[1])

            # Resize image
            pil_img = Image.fromarray(np.uint8(img)).convert("RGB")
            img_r = np.array(pil_img.resize(TARGET_SIZE[::-1], Image.BILINEAR), dtype=np.float32)
            img_r = img_r / 255.0

            # Resize mask
            pil_mask = Image.fromarray(mask * 255)
            mask_r = np.array(pil_mask.resize(TARGET_SIZE[::-1], Image.NEAREST), dtype=np.float32)
            mask_r = (mask_r >= 128).astype(np.float32)[..., np.newaxis]

            yield img_r, mask_r
    return gen


train_ds = tf.data.Dataset.from_generator(
    make_generator(train_idx),
    output_signature=(
        tf.TensorSpec(shape=(*TARGET_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(*TARGET_SIZE, 1), dtype=tf.float32),
    )
).shuffle(1500).map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(8).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    make_generator(val_idx),
    output_signature=(
        tf.TensorSpec(shape=(*TARGET_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(*TARGET_SIZE, 1), dtype=tf.float32),
    )
).batch(8).prefetch(tf.data.AUTOTUNE)


Train: 8326, Val: 2082


In [18]:
tf.data.Dataset.save(train_ds, '10000_data/train_dataset')

In [20]:
tf.data.Dataset.save(val_ds, '10000_data/val_dataset')